# Notebook: `05. Data Quality`
---

## Descrição

Notebook com a análise exploratória dos dados para definição dos filtros de qualidade a serem aplicados na camada `silver`.

Definição do catálogo e schema usado.

In [0]:
%sql
USE mvp.bronze

### Análises Yellow Taxi

Avaliado a distribuição de anos da data de pickup. Nota-se que existem registros fora do intervalo esperado (2016 - 2026).
Esses registros correspondem a cerca de 0,0004% do total de registros.

In [0]:
%sql
SELECT 
    YEAR(tpep_pickup_datetime) AS Ano,
    COUNT(*)
    FROM yellowtaxitripdata
    GROUP BY YEAR(tpep_pickup_datetime)
    ORDER BY YEAR(tpep_pickup_datetime) ASC

Ano,COUNT(*)
2001,28
2002,498
2003,50
2004,1
2007,1
2008,773
2009,1309
2010,1
2011,4
2012,1


In [0]:
%sql
SELECT
    ( SUM(CASE WHEN YEAR(tpep_pickup_datetime) < 2016-01-01
              OR YEAR(tpep_pickup_datetime) > 2026
         THEN 1 ELSE 0 END) / COUNT(*) ) * 100 AS Porcentagem_Fora_De_Periodo
FROM yellowtaxitripdata;

Porcentagem_Fora_De_Periodo
4.0147466864370505E-4


Avaliada a existência de registros com data de desembarque anterior a data de embarque.

In [0]:
%sql
SELECT
    COUNT(*)
    FROM yellowtaxitripdata
    WHERE tpep_dropoff_datetime < tpep_pickup_datetime

COUNT(*)
83442


Foram encontrados 83.442 registros onde a data/hora de desembarque é anterior a data de embarque, o que é logicamente implausível, sugerindo erro de registro pelo motorista ou serviço provedor dos dados.

Avaliado registros cujo mês/ano de embarque não é condizente com o mês/ano de referência da fonte. Esse último ponto pode sugerir a existência de registros com uma data condizente com o período analisado, mas inconsistente com o período de coleta dos registros. 
Ex: Arquivo referente a outubro de 2019, com uma corrida registrada como tendo ocorrido em outubro de 2018.

In [0]:
%sql
SELECT COUNT(*)
    FROM yellowtaxitripdata
    WHERE YEAR(tpep_pickup_datetime) <> CAST(LEFT(_source_file_yearmonth, 4) AS INT)
    OR MONTH(tpep_pickup_datetime) <> CAST(RIGHT(_source_file_yearmonth, 2) AS INT)

COUNT(*)
19420


Existem 19.420 registros onde o mês e ano de embarque não são condizentes com o mês/ano de referência do arquivo parquet de origem, sugerindo registros oncorretos.

Avaliou-se se existem registros nulos para as datas de embarque e desembarque, e pode-se aferir que não existem ocorrências.

In [0]:
%sql
SELECT COUNT(*) FROM yellowtaxitripdata
    WHERE tpep_dropoff_datetime IS NULL
    OR tpep_pickup_datetime IS NULL

COUNT(*)
0


Analisado a distribuição da distância, em milhas, percorrida pelos taxis amarelos.
Primeiramente foi avaliados os valores mínimos, máximos, quartis e o percentil 99% dos dados.
Pode-se inferir a existência de distâncias negativas, e que 99% de todas as corridas de taxi são de até 19 milhas.

In [0]:
df = spark.table("yellowtaxitripdata")

df.select("trip_distance").summary(
    "count", "min", "25%", "50%", "75%", "99%", "max"
).show()

+-------+-------------+
|summary|trip_distance|
+-------+-------------+
|  count|    674513291|
|    min|    -37264.53|
|    25%|          1.0|
|    50%|          1.7|
|    75%|          3.2|
|    99%|         19.3|
|    max| 1.90726288E7|
+-------+-------------+



Em sequência, foi avaliado a contagem de corridas com distância zero (experado, pois nas análises preliminares, o campo trip_distance pode ser double ou long, sendo que o último trunca os campos decimais) e corridas acima de 100 milhas.

In [0]:
from pyspark.sql.functions import col

negative_count = df.filter(col("trip_distance") < 0).count()
zero_count = df.filter(col("trip_distance") == 0).count()
extreme_count = df.filter(col("trip_distance") > 100).count()  # ajuste o limiar

print(f"Negativas: {negative_count:,}")
print(f"Zero: {zero_count:,}")
print(f"Acima de 100 milhas: {extreme_count:,}")

Negativas: 11,440
Zero: 7,806,274
Acima de 100 milhas: 13,105


O baixo número de corridas com distância superior a 100 milhas (13.105 registros), frente a uma base com histórico de 10 anos e milhões de linhas, indica que esses registros podem ser outliers por erro de preenchimento, ou que viagens genuínas com essas distâncias são despreziveis.
Foi também visto que existem cerca de 11.440 registros com distância negativa, indicando um registro incorreto, e 7.806.274 registros de distância zero. A distância zero pode ser explicada por arquivos .parquet da fonte que possuíam o campo "trip_distance" com o tipo long, ou seja, os decimais já viram truncados na origem, e não são recuperáveis.

Também foi realizada a análise do valor registrado pelo taxímetro (fare_amount) e valor total pago pelo passageiro (total_amount) para viagens em que a distância percorrida é zero. Além de valores de tarifas negativas, existem também outliers a partir do percentil 99 que são incompatíveis com uma viagem cuja distância percorrida seja inferior a uma milha, como os 140 doláres de fare_amount, sugerindo a possibilidade de que sejam registros incorretos.

In [0]:

zero_distance = df.filter(col("trip_distance") == 0)

zero_distance.select("fare_amount", "total_amount").summary(
    "count", "min", "25%", "50%", "75%", "90%", "95%", "99%", "99.99%", "max"
).show()

+-------+-------------+------------+
|summary|  fare_amount|total_amount|
+-------+-------------+------------+
|  count|      7806274|     7806274|
|    min|-1.33391414E8|     -1871.8|
|    25%|          2.5|        5.37|
|    50%|         13.0|       17.83|
|    75%|        33.01|       38.76|
|    90%|        60.61|       73.16|
|    95%|         75.0|       90.19|
|    99%|        140.0|      153.88|
| 99.99%|        608.0|       701.0|
|    max|     943274.8|  1084772.17|
+-------+-------------+------------+



In [0]:
# Quantas têm fare_amount > 0 apesar de distância zero?
zero_and_charged = zero_distance.filter(col("fare_amount") == 0).count()
zero_total = zero_distance.count()
print(f"{zero_and_charged:,} de {zero_total:,} corridas com 0 milhas tiveram cobrança igual 0")

103,222 de 7,806,274 corridas com 0 milhas tiveram cobrança igual 0


Apenas 103.222 corridas, de um total de 7.806.274 com distância igual a zero tiveram um valor do taxímetro registrado como também sendo zero.

Foi realizada a avaliação da distribuição de gorjetas. Esse é um campo que pode ser nulo ou zero, caso o passageiro não tenha pago gorjeta, ou tenha sido em dinheiro.

In [0]:
df.select("tip_amount", "fare_amount", "total_amount").summary(
    "count", "min", "25%", "50%", "75%", "99%", "99.99%", "max"
).show()

+-------+--------------+-------------+------------+
|summary|    tip_amount|  fare_amount|total_amount|
+-------+--------------+-------------+------------+
|  count|     674513291|    674513291|   674513291|
|    min|       -493.22|-1.33391414E8|     -2567.8|
|    25%|           0.0|          7.0|        10.3|
|    50%|           1.7|         10.0|       14.75|
|    75%|           3.0|         16.5|        22.3|
|    99%|         14.75|         70.0|        89.7|
| 99.99%|         56.92|        250.0|      297.55|
|    max|1.3339136353E8|    998310.03|  1084772.17|
+-------+--------------+-------------+------------+



Existem registros com valores de gorjeta pagos negativos, além de outliers. A grande maioria dos registros possuem valores coerentes de gorjeta, avaliado até o percentil 99, que registrou um valor de USD 56,92 dólares.
Os valores de tarifas base e total pago pelo passageiro variam de mínimos negativos, o que seriam valores impossíveis, a até 250 dólares de tarifa base, e 297,55 dólares de valores totais. Cerca de 67.451 registros possuem valores superiores a essas cifras, chegando a um valor máximo de USD 998.310 dólares de tarifa base de taxi, o que seria uma quantia irreal.

Foi avaliado se existem registros fora do intervalo aceito de métodos de pagamento (payment_type entre 0 e 6) e código de tarifa base (ratecodeID entre 1 a 6 e 99), e existem registros onde o campo ratecodeID se encontra nulo, o que não é previsto no dicionário de dados.

In [0]:
%sql
SELECT DISTINCT payment_type
    FROM yellowtaxitripdata
    ORDER BY payment_type

payment_type
0
1
2
3
4
5


In [0]:
%sql
SELECT DISTINCT ratecodeid
    FROM yellowtaxitripdata
    ORDER BY ratecodeid

ratecodeid
null
1.0
2.0
3.0
4.0
5.0
6.0
99.0


Para o campo `payment_type`, todos os registros estão dentro do intervalo esperado do dicionário de dados fornecido pela TLC, porém o campo `ratecodeid` possuí a presença de valores nulos, o que está fora do intervalo esperado, e não é um valor válido. 

Para encerrar a análise exploratória, foi avaliado se existem registros cujo ID do local de embarque e desembarque (`PULocationID`, `DOLocationID`, respectivamente) são nulos ou fora do intervalo esperado (1 a 265) indicado na tabela `taxi_zone_lookup` fornecida pela TLC. Verificou-se que não existem registros com ID de local inválido ou nulo.

In [0]:
%sql
SELECT COUNT(*)
    FROM yellowtaxitripdata
    WHERE PULocationID IS NULL
    OR DOLocationID IS NULL
    OR (PULocationID > 265 OR PULocationID < 1 )
    OR (DOLocationID > 265 OR DOLocationID < 1)

COUNT(*)
0


Avaliado a quantidade de corridas nas quais não são conhecidas (código de local 264), e o código de tarifa não é do tipo desconhecido, e onde o pagamento não é do tipo disputado, anulado ou desconhecido. Entende-se que esse tipo de viagem não teve uma causa especial explicável pelos demais campos para os locais de início e fim serem desconhecidos, e esses tipos de registros não respondem a uma das nossas perguntas de negócio, que seria identificar os locais de gargalo em horários de pico para embarque e desembarque.

In [0]:
%sql
SELECT COUNT(*)
    FROM yellowtaxitripdata
    WHERE PULocationID = 264 AND DOLocationID = 264
    AND RatecodeID <> 99
    AND payment_type NOT IN (4, 5, 6)

COUNT(*)
5663646


Existem 5.663.646 de registros que atendem a esse critério, correspondente a cerca de 0,84% do total de registros.

Avaliada a existência de viagens com início e fim fora de Nova Iorque, Início e fim desconhecido, início e fim nulo, e códigos de local de embarque/desembarque fora dos parâmetros esperados (1-265)

In [0]:
from pyspark.sql.functions import col

df = spark.table('yellowtaxitripdata')

df_foraNYC = df.filter( ( col("PUlocationID") == 265 ) & ( col("DOlocationID") == 265) & (col("RatecodeID") != 5))
df_inicioFimDesconhecido = df.filter( ( col("PUlocationID") == 264 ) & ( col("DOlocationID") == 264))
df_LocalNulo = df.filter( ( col("PUlocationID").isNull() ) | ( col("DOlocationID").isNull()) )
df_localForaLimite = df.filter( ( col("PUlocationID") < 1 ) | ( col("DOlocationID") < 1 )  | ( col("PUlocationID") > 265 ) | ( col("DOlocationID") > 265 ) )

print(f'Viagens com início e fim fora de NYC: {df_foraNYC.count():,}')
print(f'Viagens com inicio e fim desconhecidos: {df_inicioFimDesconhecido.count():,}')
print(f'Viagens com início ou fim nulos: {df_LocalNulo.count():,}')
print(f'Viagens com início e fim fora dos parametros: {df_localForaLimite.count():,}')

Viagens com início e fim fora de NYC: 74,974
Viagens com inicio e fim desconhecidos: 5,703,488
Viagens com início ou fim nulos: 0
Viagens com início e fim fora dos parametros: 0


Avaliação da distribuição de fare_amount

In [0]:
from pyspark.sql.functions import col, try_divide

df = spark.table("yellowtaxitripdata")

df = (
    df
    .filter( col("trip_distance") > 0)
    .withColumn("cost_per_mile", try_divide(col("fare_amount"), col("trip_distance")))
)

df.select("fare_amount").summary(
    "count", "min", "25%", "50%", "75%", "99%", "99.9%", "99.99%", "max"
).show()

+-------+-----------+
|summary|fare_amount|
+-------+-----------+
|  count|  666695577|
|    min|    -2564.0|
|    25%|        7.0|
|    50%|       10.0|
|    75%|       16.5|
|    99%|       70.0|
|  99.9%|     102.88|
| 99.99%|      245.9|
|    max|  998310.03|
+-------+-----------+



#### Regras de filtros para ingestão na camada silver - tabela yellowtaxitripdata

A partir dessa análise exploratória dos dados da tabela `yellowtaxitripdata` da camada `bronze`, entende-se necessário os seguintes filtros de qualidade para a construção da tabela na camada `silver`:

* Remoção de registros onde `tpep_dropoffdatetime` é anterior a `tpep_pickup_datetime`
* Remoção de registros onde o mês e ano de `tpep_pickup_datetime` não é condizente com o mês e ano de `_source_file_yearmonth`
* Remoção de registros cujo valor `trip_distance` e `fare_amount` sejam negativos ou a relação de `fare_amount` e `trip_distance` seja maior que o limiar de outlier segundo a regra da distância interquartis (> 30,75 dolares/ milha)
* Remoção dos registros quando `tip_amount` for negativo ou maior que 100 doláres, visto que 99,99% dos registros possuem gorjeta de até 56 dólares, e 100 dólares, embora seja um corte arbitrário, já é considerado significativamente maior que a grande maioria dos registros, mas não necessariamente implausível como gorjeta ao motorista.
* Remoção dos registros cujo campo `ratecodeid` seja nulo, pois nulos não são valores previstos pelo dicionário de dados fornecido pela TLC.
* Remoção de registros onde `PULocationID` e `DOLocationID` é desconhecido (código 264).
* Carregamento apenas dos campos `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `trip_distance`, `RatecodeID`, `PULocationID`, `DOLocationID`, `payment_type`, `fare_amount`, `tip_amount`, `_source_file_yearmonth`, sendo o último usado para controle interno de atualização incremental do conjunto de dados na camada `silver`

### Análises Green Taxi

Avaliado a distribuição de anos da data de pickup. Nota-se que existem registros fora do intervalo esperado (2016 - 2026).
Esses registros correspondem a cerca de 0,0016% do total de registros.

In [0]:
%sql
SELECT 
    YEAR(lpep_pickup_datetime) AS Ano,
    COUNT(*)
    FROM greentaxitripdata
    GROUP BY YEAR(lpep_pickup_datetime)
    ORDER BY YEAR(lpep_pickup_datetime) ASC

Ano,COUNT(*)
2008,117
2009,316
2010,348
2012,3
2016,16385541
2017,11736906
2018,8899314
2019,6300814
2020,1734166
2021,1068729


In [0]:
%sql
SELECT
    ( SUM(CASE WHEN YEAR(lpep_pickup_datetime) < 2016-01-01
              OR YEAR(lpep_pickup_datetime) > 2026
         THEN 1 ELSE 0 END) / COUNT(*) ) * 100 AS Porcentagem_Fora_De_Periodo
FROM greentaxitripdata;

Porcentagem_Fora_De_Periodo
0.001603883200190387


Foi analisado também a quantidade de registros cuja data/hora de desembarque (tpep_dropoff_datetime) é menor que a data de embarque (tpep_pickup_datetime) e registros cujo mês/ano de embarque não é condizente com o mês/ano de referência da fonte. Esse último ponto pode sugerir a existência de registros com uma data condizente com o período analisado, mas inconsistente com o período de coleta dos registros. 

In [0]:
%sql
SELECT
    COUNT(*)
    FROM greentaxitripdata
    WHERE lpep_dropoff_datetime < lpep_pickup_datetime

COUNT(*)
1927


In [0]:
%sql
SELECT COUNT(*)
    FROM greentaxitripdata
    WHERE YEAR(lpep_pickup_datetime) <> CAST(LEFT(_source_file_yearmonth, 4) AS INT)
    OR MONTH(lpep_pickup_datetime) <> CAST(RIGHT(_source_file_yearmonth, 2) AS INT)

COUNT(*)
4262


Existem 1.927 registrons onde a data/hora de desembarque é anterior a data de embarque. Ainda, existem 4.262 registros onde o mês e ano de embarque não são condizentes com o mês/ano de referência do arquivo parquet de origem, sugerindo registros oncorretos.

Avaliou-se se existem registros nulos para as datas de embarque e desembarque, e pode-se aferir que não existem ocorrências.

In [0]:
%sql
SELECT COUNT(*) FROM greentaxitripdata
    WHERE lpep_dropoff_datetime IS NULL
    OR lpep_pickup_datetime IS NULL

COUNT(*)
0


Analisado a distribuição da distância, em milhas, percorrida pelos taxis verdes.
Primeiramente foi avaliados os valores mínimos, máximos, quartis e o percentil 99%, 99,9% e 99,99% dos dados.
Pode-se inferir a existência de distâncias negativas, e que 99,9% de todas as corridas de taxi são de até 27,22 milhas.

In [0]:
df = spark.table("greentaxitripdata")

df.select("trip_distance").summary(
    "count", "min", "25%", "50%", "75%", "99%", "99.9%", "max"
).show()

+-------+-------------+
|summary|trip_distance|
+-------+-------------+
|  count|     49255457|
|    min|    -20329.08|
|    25%|         1.05|
|    50%|          1.9|
|    75%|         3.69|
|    99%|        17.49|
|  99.9%|        27.22|
|    max|    360068.14|
+-------+-------------+



Em sequência, foi avaliado a contagem de corridas com distância zero (experado, pois nas análises preliminares, o campo trip_distance pode ser double ou long, sendo que o último trunca os campos decimais) e corridas acima de 100 milhas.

In [0]:
from pyspark.sql.functions import col

negative_count = df.filter(col("trip_distance") < 0).count()
zero_count = df.filter(col("trip_distance") == 0).count()
extreme_count = df.filter(col("trip_distance") > 100).count()

print(f"Negativas: {negative_count:,}")
print(f"Zero: {zero_count:,}")
print(f"Acima de 100 milhas: {extreme_count:,}")

Negativas: 19,527
Zero: 857,634
Acima de 100 milhas: 4,638


O baixo número de corridas com distância superior a 100 milhas (4.638 registros), frente a uma base com histórico de 10 anos e milhões de linhas, indica que esses registros podem ser outliers por erro de preenchimento, ou que viagens genuínas com essas distâncias são despreziveis.
Foi também visto que existem cerca de 19.527 registros com distância negativa, indicando um registro incorreto, e 857.634 registros de distância zero. A distância zero pode ser explicada por arquivos .parquet da fonte que possuíam o campo "trip_distance" com o tipo double, ou seja, os decimais já viram truncados na origem, e não são recuperáveis.

Também foi realizada a análise do valor registrado pelo taxímetro (fare_amount) e valor total pago pelo passageiro (total_amount) para viagens em que a distância percorrida é zero. Além de valores de tarifas negativas, existem também outliers a partir do percentil 99 que são incompatíveis com uma viagem cuja distância percorrida seja inferior a uma milha, como os 95 doláres de fare_amount, sugerindo a possibilidade de que sejam registros incorretos.

In [0]:
zero_distance = df.filter(col("trip_distance") == 0)

zero_distance.select("fare_amount", "total_amount").summary(
    "count", "min", "25%", "50%", "75%", "90%", "95%", "99%", "max"
).show()

+-------+-----------+------------+
|summary|fare_amount|total_amount|
+-------+-----------+------------+
|  count|     857634|      857634|
|    min|     -890.0|      -890.3|
|    25%|        2.5|         4.3|
|    50%|       10.0|        11.3|
|    75%|       22.2|        24.8|
|    90%|       40.2|        46.0|
|    95%|       52.0|        57.3|
|    99%|       95.0|       100.0|
|    max|     4307.0|     8999.91|
+-------+-----------+------------+



Apenas 26.479 corridas, de um total de 857.634 com distância igual a zero tiveram um valor do taxímetro registrado como também sendo zero.

In [0]:
df.select("fare_amount", "total_amount").summary(
    "count", "min", "25%", "50%", "75%", "90%", "95%", "99%", "99.9%", "99.99%", "max"
).show()

+-------+-----------+------------+
|summary|fare_amount|total_amount|
+-------+-----------+------------+
|  count|   49255457|    49255457|
|    min|     -890.0|      -890.3|
|    25%|        6.5|         8.3|
|    50%|       10.0|       12.25|
|    75%|       16.5|       19.57|
|    90%|       26.0|       30.85|
|    95%|       35.0|        40.6|
|    99%|      56.75|        65.4|
|  99.9%|       92.5|       105.8|
| 99.99%|      300.0|       254.0|
|    max|   10445.84|    10528.75|
+-------+-----------+------------+



In [0]:
# Quantas têm fare_amount > 0 apesar de distância zero?
zero_and_charged = zero_distance.filter(col("fare_amount") == 0).count()
zero_total = zero_distance.count()
print(f"{zero_and_charged:,} de {zero_total:,} corridas com 0 milhas tiveram cobrança igual 0")

26,479 de 857,634 corridas com 0 milhas tiveram cobrança igual 0


Foi realizada a avaliação da distribuição de gorjetas. Esse é um campo que pode ser nulo ou zero, caso o passageiro não tenha pago gorjeta, ou tenha sido em dinheiro.

In [0]:
from pyspark.sql.functions import col, try_divide

df = spark.table("greentaxitripdata")

df = (
    df
    .filter( col("trip_distance") > 0)
    .withColumn("cost_per_mile", try_divide(col("fare_amount"), col("trip_distance")))
)

df.select("fare_amount").summary(
    "count", "min", "25%", "50%", "75%", "99%", "99.9%", "99.99%", "max"
).show()

+-------+-----------+
|summary|fare_amount|
+-------+-----------+
|  count|   48378296|
|    min|     -480.0|
|    25%|        6.5|
|    50%|       10.0|
|    75%|       16.3|
|    99%|      56.01|
|  99.9%|       89.5|
| 99.99%|      288.6|
|    max|   10445.84|
+-------+-----------+



Novamente, identificou-se a existência de registros negativos, e um outlier de 900 dólares. Embora seja importante pagar uma boa gorjeta ao motorista por um bom serviço, não é razoável considerar que esse tenha sido o valor de fato pago, considerando que o código abaixo mostra que a corrida teve um fare_amount de USD 19,50 dólares e com 2,9 milhas de distância.

In [0]:
%sql
SELECT fare_amount, trip_distance
    FROM greentaxitripdata
    WHERE tip_amount = 900


fare_amount,trip_distance
19.5,2.9


Foi avaliado se existem registros fora do intervalo aceito de métodos de pagamento (payment_type entre 0 e 6), e existem registros onde o campo se encontra nulo, o que não é previsto no dicionário de dados.

In [0]:
%sql
SELECT DISTINCT payment_type
    FROM greentaxitripdata
    ORDER BY payment_type

payment_type
null
1.0
2.0
3.0
4.0
5.0


Há 1.966.046 de registros com método de pagamento nulo.

In [0]:
%sql
SELECT COUNT(*)
    FROM greentaxitripdata
    WHERE payment_type IS NULL

COUNT(*)
1966046


A mesma análise foi feita para o campo ratecodeid, que deve ter um intervalo de 1 a 6, e 99. Foram identificados registros com o campo nulo.

In [0]:
%sql
SELECT DISTINCT ratecodeid
    FROM greentaxitripdata
    ORDER BY ratecodeid

ratecodeid
null
1.0
2.0
3.0
4.0
5.0
6.0
99.0


Novamente, também existem os mesmos 1.966.046 registros com ratecodeid nulo. Podemos considerá-los registros inválidos.

In [0]:
%sql
SELECT COUNT(*) ratecodeid
    FROM greentaxitripdata
    WHERE RatecodeID IS NULL

ratecodeid
1966046


Para encerrar a análise exploratória, foi avaliado se existem registros cujo ID do local de embarque e desembarque (`PULocationID`, `DOLocationID`, respectivamente) são nulos ou fora do intervalo esperado (1 a 265) indicado na tabela `taxi_zone_lookup` fornecida pela TLC. Verificou-se que não existem registros com ID de local inválido ou nulo.

In [0]:
from pyspark.sql.functions import col

df = spark.table('greentaxitripdata')

df_foraNYC = df.filter( ( col("PUlocationID") == 265 ) & ( col("DOlocationID") == 265) & (col("RatecodeID") != 5))
df_inicioFimDesconhecido = df.filter( ( col("PUlocationID") == 264 ) & ( col("DOlocationID") == 264))
df_LocalNulo = df.filter( ( col("PUlocationID").isNull() ) | ( col("DOlocationID").isNull()) )
df_localForaLimite = df.filter( ( col("PUlocationID") < 1 ) | ( col("DOlocationID") < 1 )  | ( col("PUlocationID") > 265 ) | ( col("DOlocationID") > 265 ) )

print(f'Viagens com início e fim fora de NYC: {df_foraNYC.count():,}')
print(f'Viagens com inicio e fim desconhecidos: {df_inicioFimDesconhecido.count():,}')
print(f'Viagens com início ou fim nulos: {df_LocalNulo.count():,}')
print(f'Viagens com início e fim fora dos parametros: {df_localForaLimite.count():,}')

Viagens com início e fim fora de NYC: 5,582
Viagens com inicio e fim desconhecidos: 35,289
Viagens com início ou fim nulos: 0
Viagens com início e fim fora dos parametros: 0


#### Regras de filtros para ingestão na camada silver - tabela greentaxitripdata

A partir dessa análise exploratória dos dados da tabela `greentaxitripdata` da camada `bronze`, entende-se necessário os seguintes filtros de qualidade para a construção da tabela na camada `silver`:

* Remoção de registros onde `lpep_dropoffdatetime` é anterior a `lpep_pickup_datetime`
* Remoção de registros onde o mês e ano de `lpep_pickup_datetime` não é condizente com o mês e ano de `_source_file_yearmonth`
* Remoção dos registros quando `tip_amount` for negativo ou maior de 100 doláres, visto que 99,99% dos registros possuem gorjeta de até 62 dólares, e 100 dólares, embora seja um corte arbitrário, já é considerado significativamente maior que a grande maioria dos registros, mas não necessariamente implausível como gorjeta ao motorista, dado os valores encontrados.
* Remoção de registros quando o campo `fare_amount` e `trip_miles` são negativos ou a relação entre `fare_amount` e `trip_miles` esteja fora da regra de outliers da distância interquartis (> 31 dólares por milha)
* Remoção de registros com `PULocationID` e `DOLocationID` desconhecidos (código 264).
* Remoção dos registros cujo campo `payment_type` e `ratecodeid` sejam nulos, pois nulos não são valores previstos pelo dicionário de dados fornecido pela TLC.

### Análises For Hire Vehicles

Assim como na base de taxis verdes e amarelos, foi avaliada a distribuição de anos da data de pickup. Não há registros fora do intervalo esperado (2016 - 2026)

In [0]:
%sql
SELECT 
    YEAR(pickup_datetime) AS Ano,
    COUNT(*)
    FROM forhirevehicles
    GROUP BY YEAR(pickup_datetime)
    ORDER BY YEAR(pickup_datetime) ASC

Ano,COUNT(*)
2016,132114083
2017,192309557
2018,260874754
2019,43261276
2020,14945465
2021,14805265
2022,14511664
2023,15858639
2024,17630326
2025,25047544


Foi analisado também a quantidade de registros cuja data/hora de desembarque (pickup_datetime) é menor que a data de embarque (dropoff_datetime) e registros cujo mês/ano de embarque não é condizente com o mês/ano de referência da fonte. Esse último ponto pode sugerir a existência de registros com uma data condizente com o período analisado, mas inconsistente com o período de coleta dos registros.

In [0]:
%sql
SELECT
    COUNT(*)
    FROM forhirevehicles
    WHERE dropOff_datetime < pickup_datetime

COUNT(*)
204928012


In [0]:
%sql
SELECT COUNT(*)
    FROM forhirevehicles
    WHERE YEAR(pickup_datetime) <> CAST(LEFT(_source_file_yearmonth, 4) AS INT)
    OR MONTH(pickup_datetime) <> CAST(RIGHT(_source_file_yearmonth, 2) AS INT)

COUNT(*)
120


Existem 204.928.012 de registros onde a data/hora de desembarque é anterior à data/hora de embarque, e apenas 120 registros onde o mês/ano de embarque não é coerente conm o período de apuração da fonte de dados.

In [0]:
%sql
SELECT COUNT(*) FROM forhirevehicles
    WHERE dropOff_datetime IS NULL
    OR pickup_datetime IS NULL

COUNT(*)
0


Não existem registros nulos para as datas de embarque e desembarque

Foi avaliado se existem bases de despache com vínculo a uma empresa de transporte de aplicativos, utilizando uma tabela de consulta criada a partir de informações do guia de uso da base pela TLC, e avaliado a data máxima em que esses registros estão presentes nessa tabela. A TLC informa que a partir de 2019 os registros de viagens realizadas pelas empresas de transporte por aplicativo com volume superior a 10 mil viagens por dia passariam a ser armazenadas na tabela forhirevehicleshighvolume.

In [0]:
%sql
SELECT DISTINCT 
    dispatching_base_num, 
    lkup.App_Company_Affiliation FROM forhirevehicles fhv
    INNER JOIN fhv_base_lookup lkup
        on lkup.License_Number = fhv.dispatching_base_num 

    ORDER BY dispatching_base_num ASC

dispatching_base_num,App_Company_Affiliation
B02404,Uber
B02510,Lyft
B02512,Uber
B02617,Uber
B02682,Uber
B02864,Uber
B02865,Uber
B02866,Uber
B02869,Uber
B02870,Uber


In [0]:
%sql
SELECT 
    MAX(pickup_datetime)
    FROM forhirevehicles fhv
    INNER JOIN fhv_base_lookup lkup
        on lkup.License_Number = fhv.dispatching_base_num 
    

MAX(pickup_datetime)
2019-01-31T23:59:59.000


O último registro feito pelas empresas de viagem por aplicativo na base forhirevehicles foi feito em 31 de janeiro de 2019, não sendo então necessário o consumo de dados posterior a esse período para a camada silver.

In [0]:
%sql
SELECT PUlocationID, count(*)
    FROM forhirevehicles
    GROUP BY PUlocationID

PUlocationID,count(*)
147.0,758222
185.0,1014186
247.0,1767301
151.0,2014624
198.0,2931124
176.0,137570
253.0,66776
150.0,636106
73.0,514054
74.0,3203927


In [0]:
df = spark.table('forhirevehicles')

df_foraNYC = df.filter( ( col("PUlocationID") == 265 ) & ( col("DOlocationID") == 265))
df_inicioFimDesconhecido = df.filter( ( col("PUlocationID") == 264 ) & ( col("DOlocationID") == 264))
df_LocalNulo = df.filter( ( col("PUlocationID").isNull() ) | ( col("DOlocationID").isNull()) )
df_localForaLimite = df.filter( ( col("PUlocationID") < 1 ) | ( col("DOlocationID") < 1 )  | ( col("PUlocationID") > 265 ) | ( col("DOlocationID") > 265 ) )

print(f'Viagens com início e fim fora de NYC: {df_foraNYC.count():,}')
print(f'Viagens com início ou fim nulos: {df_LocalNulo.count():,}')
print(f'Viagens com início e fim fora dos parametros: {df_localForaLimite.count():,}')

Viagens com início e fim fora de NYC: 936,690
Viagens com início ou fim nulos: 202,022,265
Viagens com início e fim fora dos parametros: 2,348,590


#### Regras de filtros para ingestão na camada silver - tabela forhirevehicles

A partir dessa análise exploratória dos dados da tabela `forhirevehicles` da camada `bronze`, entende-se necessário os seguintes filtros de qualidade para a construção da tabela na camada `silver`:

* Remoção de registros onde `dropoff_datetime` é anterior a `pickup_datetime`
* Remoção de registros onde o mês e ano de `pickupdatetime` não é condizente com o mês e ano de `_source_file_yearmonth`
* Remoção de registros onde `PUlocationID` e `DOlocationID` sejam nulos, fora do intervalo de 1 a 265, ou ambos sejam iguais a 264 ou 265. A remoção de `DOlocationID` ocorre apenas a partir de 2018. Para periodos anteriores se considera válido pois não era um item obrigatório pela TLC para a categoria.
* Ingestão dos registros apenas até o mês de janeiro de 2019, visto que a partir de fevereiro de 2019 as viagens de empresas de transporte por aplicativo de alto volume (Uber, Lyft, Via e Juno) passaram a ser reportadas separadamente na categoria High-Volume For-Hire Services (HVFHS), conforme a Local Law 149 de 2018, que entrou em vigor em 1º de fevereiro de 2019. Essas viagens são tratadas na tabela `fhvhv_tripdata`, e portanto não interessam à tabela `forhirevehicles` a partir dessa data.

### Análises For Hire Services High Volume

Avaliado a distribuição de anos da data de pickup.
Não existem registros fora do intervalo esperado (2019 - 2026).

In [0]:
%sql
SELECT 
    YEAR(pickup_datetime) AS Ano,
    COUNT(*)
    FROM forhirevehicleshighvolume
    GROUP BY YEAR(pickup_datetime)
    ORDER BY YEAR(pickup_datetime) ASC

Ano,COUNT(*)
2019,234630264
2020,143309871
2021,174596652
2022,212416083
2023,232490020
2024,239470448
2025,243589684
2026,126771982


Foi analisado também a quantidade de registros cuja data/hora de desembarque (dropoff_datetime) é menor que a data de embarque (pickup_datetime) e registros cujo mês/ano de embarque não é condizente com o mês/ano de referência da fonte. Esse último ponto pode sugerir a existência de registros com uma data condizente com o período analisado, mas inconsistente com o período de coleta dos registros. 

In [0]:
%sql
SELECT
    COUNT(*)
    FROM forhirevehicleshighvolume
    WHERE dropOff_datetime < pickup_datetime

COUNT(*)
61836


In [0]:
%sql
SELECT COUNT(*)
    FROM forhirevehicleshighvolume
    WHERE YEAR(pickup_datetime) <> CAST(LEFT(_source_file_yearmonth, 4) AS INT)
    OR MONTH(pickup_datetime) <> CAST(RIGHT(_source_file_yearmonth, 2) AS INT)

COUNT(*)
0


Existem 61.836 registros de viagens com data de desembarque anterior a de embarque, e não há registros onde o mês e ano do embarque é inconsistente com o mês e ano de coleta do registro de viagem.

Avaliou-se se existem registros nulos para as datas de embarque e desembarque, e pode-se aferir que não existem ocorrências.

In [0]:
%sql
SELECT COUNT(*) FROM forhirevehicleshighvolume
    WHERE dropoff_datetime IS NULL
    OR pickup_datetime IS NULL

COUNT(*)
0


Analisado a distribuição da distância, em milhas, percorrida pelos veículos.
Primeiramente foi avaliados os valores mínimos, máximos, quartis, o percentil 99% e 99,99% dos dados.
Pode-se verificar que não existem distâncias negativas, como observado em outras bases, mas há registros de viagens com distância igual a zero (possivelmente viagens canceladas, visto que todos os arquivos .parquet que compõem do dataset possuem o campo `trip_miles` como sendo tipo double), e 99,99% das viagens são de até 95,11 milhas.
Existe um outlier com 5.380,78 milhas de distância, que pode ser desconsiderado, visto que a suposta distância é quase o dobro da distância entre as costas oeste e leste dos EUA, complementado pelo fato que esse registro particular teve duração de cerca de uma hora.

In [0]:
df = spark.table("forhirevehicleshighvolume")

df.select("trip_miles").summary(
    "count", "min", "25%", "50%", "75%", "99.99%", "max"
).show()

+-------+----------+
|summary|trip_miles|
+-------+----------+
|  count|1607275004|
|    min|       0.0|
|    25%|      1.57|
|    50%|      2.96|
|    75%|     6.093|
| 99.99%|     95.11|
|    max|   5380.78|
+-------+----------+



In [0]:
%sql
SELECT trip_miles, pickup_datetime, dropoff_datetime from forhirevehicleshighvolume
WHERE trip_miles = 5380.78

trip_miles,pickup_datetime,dropoff_datetime
5380.78,2025-10-09T18:41:06.000,2025-10-09T19:44:40.000


Em sequência, foi avaliado a contagem de corridas com distância zero e corridas acima de 100 milhas.

In [0]:
from pyspark.sql.functions import col

zero_count = df.filter(col("trip_miles") == 0).count()
extreme_count = df.filter(col("trip_miles") > 100).count()  # ajuste o limiar

print(f"Negativas: {negative_count:,}")
print(f"Zero: {zero_count:,}")
print(f"Acima de 100 milhas: {extreme_count:,}")

Negativas: 0
Zero: 882,487
Acima de 100 milhas: 155,922


Complementando a análise de corridas com distância superior a 100 milhas, validou-se se existem registros de viagens com local de embarque/desembarque desconhecido ou fora de Nova Iorque (códigos `264` e `265`), e foi verificado que existem cerca de 63.080.445 registros.

In [0]:
%sql
SELECT COUNT(*)
    FROM forhirevehicleshighvolume
    WHERE PUlocationID IN (265, 264)
    OR DOlocationID IN (265, 264)

COUNT(*)
63080445


 Como é apenas de interesse das perguntas de negócio apenas viagens que ocorreram dentro de Nova Iorque, foi validado também se houveram viagens com início e fim fora da cidade, ou com início e fim desconhecido.

In [0]:
df = spark.table('forhirevehicleshighvolume')

df_foraNYC = df.filter( ( col("PULocationID") == 265 ) & ( col("DOLocationID") == 265))
df_inicioFimDesconhecido = df.filter( ( col("PULocationID") == 264 ) & ( col("DOLocationID") == 264))

print(f'Viagens com início e fim fora de NYC: {df_foraNYC.count():,}')
print(f'Viagens com início e fim desconhecido: {df_inicioFimDesconhecido.count():,}')

Viagens com início e fim fora de NYC: 27,701
Viagens com início e fim desconhecido: 1,938


Pode-se verificar que 27.701 viagens não tiveram início ou fim na cidade de Nova Iorque, não sendo então registros válidos para responder as perguntas de negócio do MVP, e 1.938 registros tiveram início e fim desconhecido, também não sendo úteis para responder as perguntas de negócio.

Já para as viagens com distância igual a 0 milhas, foi feita uma análise da distribuição do tempo de viagem, em segundos, e da tarifa base cobrada ao passageiro.

In [0]:
from pyspark.sql.functions import col

df = spark.table("forhirevehicleshighvolume")
df_filtrado = df.filter(col("trip_miles") == 0)


df_filtrado.select("base_passenger_fare", "trip_time").summary(
    "count", "min", "25%", "50%", "75%", "99.99%", "max"
).show()

+-------+-------------------+---------+
|summary|base_passenger_fare|trip_time|
+-------+-------------------+---------+
|  count|             882487|   882487|
|    min|            -150.85|        0|
|    25%|               7.05|      278|
|    50%|               9.74|      690|
|    75%|               16.7|     1333|
| 99.99%|              454.4|    39131|
|    max|             971.32|    83324|
+-------+-------------------+---------+



E para finalizar, o custo da tarifa base cobrada quando o tempo da viagem foi de zero segundos.

In [0]:
df = spark.table("forhirevehicleshighvolume")
df_filtrado = df.filter( (col("trip_miles") == 0 ) & (col("trip_time") == 0 ))


df_filtrado.select("base_passenger_fare").summary(
    "count", "min", "25%", "50%", "75%", "99.99%", "max"
).show()

+-------+-------------------+
|summary|base_passenger_fare|
+-------+-------------------+
|  count|              13972|
|    min|             -27.75|
|    25%|                5.3|
|    50%|               9.59|
|    75%|              20.62|
| 99.99%|             443.84|
|    max|             839.89|
+-------+-------------------+



A análise das viagens da base HVFHS com distância superior a 100 milhas identificou 155.922 registros, um volume ínfimo frente a uma base de aproximadamente 1,5 bilhão de viagens, o que sugere que se trata de outliers decorrentes de erro de preenchimento. Como verificação adicional, testou-se a hipótese de que essas viagens teriam origem ou destino fora da cidade de Nova Iorque, o que justificaria as distâncias elevadas. Para isso, verificou-se se os códigos de localização de embarque e desembarque correspondiam às categorias "Desconhecido" (264) ou "Fora de Nova Iorque" (265), conforme a legenda da tabela `taxi_zone_lookup`. Nenhum registro se enquadrou nessas categorias, o que afasta a hipótese e permite descartar as viagens acima de 100 milhas como outliers.

Já as viagens com distância igual a zero foram analisadas sob a hipótese de que representariam corridas canceladas. Diferentemente das bases de táxis amarelos e verdes, que possuem o campo `payment_type` e permitem identificar viagens anuladas, a base HVFHS não dispõe de um indicador equivalente. Por isso, a investigação recorreu à distribuição dos valores de tarifa base do passageiro (`base_passenger_fare`) e do tempo de viagem (`trip_time`) nesses registros.

Entre as viagens com distância zero, observaram-se tarifas base negativas (mínimo de -150 dólares) e valores de até 971 dólares. O campo mais inconsistente com uma viagem sem deslocamento foi o `trip_time`, que variou de 0 segundo, com 3º quartil em 1.333 segundos, a um máximo de 83.324 segundos. Ao restringir a análise às viagens com distância zero e tempo de viagem zero, restaram 13.972 registros, um número reduzido para um período de sete anos e 1,5 bilhão de viagens. Ainda assim, esses registros apresentaram valores cobrados negativos, 3º quartil de 20,62 dólares e outliers de até 839,89 dólares, o que impede caracterizá-los de forma consistente como cancelamentos.

Diante disso, optou-se pela exclusão de todos os registros com distância igual a zero (882.487 registros, ou 0,054% do dataset). A decisão se justifica por não haver forma segura de inferir se as cobranças correspondem a taxas de cancelamento ou se as distâncias zeradas resultam de erros de coleta ou envio dos dados, uma vez que a TLC recebe os dados das operadoras em sua forma nativa e não garante sua acurácia ou veracidade. Dado o peso marginal desses registros, a exclusão é a alternativa mais pragmática e não compromete as análises.


Foi realizada a avaliação da distribuição de gorjetas. Esse é um campo que pode ser nulo ou zero, caso o passageiro não tenha pago gorjeta.

In [0]:

df = spark.table("forhirevehicleshighvolume")

df.select("tips").summary(
    "count", "min", "25%", "50%", "75%", "99%", "99.99%", "max"
).show()

+-------+----------+
|summary|      tips|
+-------+----------+
|  count|1607275004|
|    min|       0.0|
|    25%|       0.0|
|    50%|       0.0|
|    75%|       0.0|
|    99%|     14.71|
| 99.99%|     53.65|
|    max|    1000.0|
+-------+----------+



A grande maioria dos passageiros não pagam gorjeta aos motoristas, sendo cerca de 1% das viagens com registros de gorjeta. 99.99% de todas as viagens possuem uma gorjeta de até 54 dólares, com um outlier de 1000 dólares.
Como já foi avaliado anteriormente que não existem registros de viagens com início ou fim fora de Nova Iorque, foi realizada uma consulta no registro que originou esse valor de gorjeta. Verificou-se que a tarifa básica ao passageiro foi de 1.493,98 dólares, que a distância percorrida foi de 409 milhas, e que os códigos de embarque e desembarque foram `132` e `265`Embora esse outlier seja significativamente superior aos demais registros de gorjeta, observa-se que a viagem se originou dentro da cidade de Nova Iorque, e encerrou fora da cidade, com duração de cerca de 7 horas, sendo então o outlier justificável.

In [0]:
%sql
SELECT 
    base_passenger_fare
    , trip_miles
    , PUlocationID
    , DOlocationID
    , trip_time/ 3600 as hora
    FROM forhirevehicleshighvolume
    WHERE tips = 1000

base_passenger_fare,trip_miles,PUlocationID,DOlocationID,hora
1493.98,409.26,132,265,7.255


Validado se todos os registros possuem código de embarque e desenbarque válido.

In [0]:
%sql
SELECT COUNT(*)
    FROM forhirevehicleshighvolume
    WHERE PULocationID IS NULL
    OR DOLocationID IS NULL
    OR (PULocationID > 265 OR PULocationID < 1 )
    OR (DOLocationID > 265 OR DOLocationID < 1)

COUNT(*)
0


Validado a distribuição de base_passenger_fare, driver_pay, trip_miles

In [0]:
from pyspark.sql.functions import col

df = spark.table("forhirevehicleshighvolume")

df.select("base_passenger_fare", "driver_pay").summary(
    "count", "min", "25%", "50%", "75%", "99.99%", "max"
).show()

+-------+-------------------+----------+
|summary|base_passenger_fare|driver_pay|
+-------+-------------------+----------+
|  count|         1607275004|1607275004|
|    min|           -1969.59|  -6867.28|
|    25%|               10.7|      8.15|
|    50%|              17.16|      13.7|
|    75%|              28.32|     23.14|
| 99.99%|             348.79|     248.8|
|    max|            8157.74|   4894.62|
+-------+-------------------+----------+



Verificado a existência de registros com tarifa base negativa, outlier de 8.157 dólares. 99,99% de todos os registros tem tarifa base de até 348,79 dólares.
Já a remuneração do motorista também possuí registros negativos, outlier de 4894,62 dólares. 99,99% de todos os registros são de até 248,8 dólares.

Como para essa base foi difícil determinar se outliers de distância e valor são genuinamente outliers, ou apenas viagens demasiado longas, avaliei a distribuição do indicador de custo por milha (USD/Mi) para identificar outliers genuínos, usando a regra IEQ ( 3° Quartil + 1.5 x (3°Quartil - 1° Quartil)), o resultado do cálculo foi usado como regra de filtro na camada silver.

In [0]:
from pyspark.sql.functions import col, try_divide

df = spark.table("forhirevehicleshighvolume")

df = (
    df
    .filter(col("trip_miles") <= 100)
    .withColumn("cost_per_mile", try_divide(col("base_passenger_fare"), col("trip_miles")))
)
    

df.select("cost_per_mile").summary(
    "count", "min", "25%", "50%", "75%", "90%", "99%", "99.9%", "99.99%", "max"
).show()

+-------+------------------+
|summary|     cost_per_mile|
+-------+------------------+
|  count|        1606236595|
|    min|          -38010.0|
|    25%| 3.988830486202365|
|    50%| 5.598369011213048|
|    75%| 7.828601472134596|
|    90%| 11.10862262038074|
|    99%|             24.09|
|  99.9%|50.641025641025635|
| 99.99%|211.72727272727272|
|    max|520399.99999999994|
+-------+------------------+



In [0]:
%sql
SELECT DISTINCT 
    hvfhs_license_num
    FROM forhirevehicleshighvolume
    ORDER BY hvfhs_license_num

hvfhs_license_num
HV0002
HV0003
HV0004
HV0005


Não há codigos de licença fora do especificado no dicionário de dados.

#### Regras de filtros para ingestão na camada silver - tabela forhirevehicles

A partir dessa análise exploratória dos dados da tabela `forhirevehicleshighvolume` da camada `bronze`, entende-se necessário os seguintes filtros de qualidade para a construção da tabela na camada `silver`:

* Remoção de registros onde `dropoff_datetime` é anterior a `pickup_datetime`
* Remoção de registros onde `base_passenger_fare` e `driver_pay` são negativos.
* Remoção de registros onde `trip_miles` é igual a 0
* Remoção de registros onde `PULocationID` e `DOLocationID` são ambos 264 ou 265.
* Remoção de registros cuja relação de `base_passenger_fare` e `trip_miles` seja superior a USD 13,58 dólares (limiar para outliers considerando a regra do IQR para outliers)

##### Tabela Auxiliar Feriados

Avaliado os primeiros 10 registros

In [0]:
%sql

SELECT 
    holiday_name,
    substring(holiday_name, ) FROM feriados_us_ny LIMIT 10

date,holiday_name
2016-01-01,New Year's Day
2016-05-30,Memorial Day
2016-07-04,Independence Day
2016-09-05,Labor Day
2016-11-24,Thanksgiving Day
2016-12-25,Christmas Day
2016-12-26,Christmas Day (observed)
2016-01-18,Martin Luther King Jr. Day
2016-02-15,Susan B. Anthony Day; Washington's Birthday
2016-10-10,Columbus Day


Avaliada a contagem total de registros

In [0]:
%sql
SELECT COUNT(*) FROM feriados_us_ny

COUNT(*)
164


Avaliada a existência de feriados com datas observadas e não observadas. Para a camada `silver`, devem ser considerados apenas datas de feriados efetivamente observados.

In [0]:
%sql
WITH flagged AS (
  SELECT
    *,
    date,
    holiday_name,
    holiday_name RLIKE '(?i)\\(observed\\)' AS is_observed,
    TRIM(REGEXP_REPLACE(holiday_name, '(?i)\\s*\\(observed\\)', '')) AS holiday_name_clean,
    YEAR(date) AS holiday_year
  FROM feriados_us_ny
),
grouped AS (
  SELECT
    *,
    MAX(CAST(is_observed AS INT)) OVER (PARTITION BY holiday_year, holiday_name_clean) AS has_observed_in_group
  FROM flagged
)
SELECT
  date,
  holiday_name,
  holiday_name_clean,
  is_observed
FROM grouped
WHERE 
    has_observed_in_group = 0 OR is_observed = TRUE

date,holiday_name,holiday_name_clean,is_observed
2016-12-26,Christmas Day (observed),Christmas Day,true
2016-10-10,Columbus Day,Columbus Day,false
2016-11-08,Election Day,Election Day,false
2016-07-04,Independence Day,Independence Day,false
2016-09-05,Labor Day,Labor Day,false
2016-02-12,Lincoln's Birthday,Lincoln's Birthday,false
2016-01-18,Martin Luther King Jr. Day,Martin Luther King Jr. Day,false
2016-05-30,Memorial Day,Memorial Day,false
2016-01-01,New Year's Day,New Year's Day,false
2016-02-15,Susan B. Anthony Day; Washington's Birthday,Susan B. Anthony Day; Washington's Birthday,false


Regra ingestão: Agrupar feriados por ano. Feriados com versão observada considera a versão observada, feriados sem versão observada mantem a versao original

#### Inflação

Avaliada a existência de registros sem valor de inflação divulgado

In [0]:
%sql
SELECT * FROM inflacao_cpi_u
    WHERE observation_date IS NULL OR CPIAUCSL IS NULL

observation_date,CPIAUCSL
2025-10-01,null


In [0]:
%sql
SELECT * FROM inflacao_cpi_t
    WHERE observation_date IS NULL OR CPITRNSL IS NULL

observation_date,CPITRNSL
2025-10-01,null


Para tratar na silver, será feita uma média entre os valores do mes seguinte e anterior aos periodos vazios, para poder ser feita uma estimativa de inflação no periodo ausente e possibilitar a correção dos valores pela inflação